In [5]:
import json
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/corporate_finance_2016_2025.json")
METADATA_PATH = Path("../data/reference/metadata_codes_0003060791.csv")
OUTPUT_PATH = Path("../data/processed/corporate_finance_long.csv")

with RAW_PATH.open(encoding="utf-8") as f:
    payload = json.load(f) # json.load(f) は、JSONファイルをPythonの辞書へ変換
# with を使うと、処理終了時にファイルを自動で閉じる
# payload は、APIレスポンス全体を持つ辞書

values = payload["GET_STATS_DATA"]["STATISTICAL_DATA"]["DATA_INF"]["VALUE"]
# 辞書を階層順にたどり、統計レコードが格納されている VALUE を取り出す
values

[{'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20160',
  '@unit': '百万円',
  '$': '293657106'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20170',
  '@unit': '百万円',
  '$': '305552557'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20180',
  '@unit': '百万円',
  '$': '317442376'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20190',
  '@unit': '百万円',
  '$': '319380812'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20200',
  '@unit': '百万円',
  '$': '337555141'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20210',
  '@unit': '百万円',
  '$': '353244936'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20220',
  '@unit': '百万円',
  '$': '367432474'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20230',
  '@unit': '百万円',
  '$': '388041999'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20240',
  '@

In [6]:
if isinstance(values, dict):
    values = [values]
# isinstance(values, dict) は、values が辞書型かどうかを確認
# 通常、複数件のデータがあるとき、values はリストだが、e-Stat APIでは取得結果が1件だけの場合、
# リストではなく辞書だけが返る場合がある。
# そのままでは、複数件を前提にした後続の処理と形が異なってしまうため、辞書をリストで包む
# このコードによって、データが1件でも複数件でも、常に「レコードのリスト」として扱えるように形を統一する。

values

[{'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20160',
  '@unit': '百万円',
  '$': '293657106'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20170',
  '@unit': '百万円',
  '$': '305552557'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20180',
  '@unit': '百万円',
  '$': '317442376'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20190',
  '@unit': '百万円',
  '$': '319380812'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20200',
  '@unit': '百万円',
  '$': '337555141'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20210',
  '@unit': '百万円',
  '$': '353244936'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20220',
  '@unit': '百万円',
  '$': '367432474'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20230',
  '@unit': '百万円',
  '$': '388041999'},
 {'@cat01': '022',
  '@cat02': '108',
  '@cat03': '25',
  '@time': '20240',
  '@

In [7]:
df = pd.DataFrame(values).rename(
    columns={
        "@cat01": "item_code",# 財務項目コード
        "@cat02": "industry_code",# 業種コード
        "@cat03": "capital_size_code",# 資本金規模コード
        "@time": "time_code",# 年度コード
        "@unit": "unit",
        "$": "amount",# 金額
    }
)
df

,item_code,industry_code,capital_size_code,time_code,unit,amount
0,022,108,25,20160,百万円,293657106
1,022,108,25,20170,百万円,305552557
2,022,108,25,20180,百万円,317442376
3,022,108,25,20190,百万円,319380812
4,022,108,25,20200,百万円,337555141
...,...,...,...,...,...,...
535,157,129,16,20210,百万円,4029975
536,157,129,16,20220,百万円,4333883
537,157,129,16,20230,百万円,3249057
538,157,129,16,20240,百万円,5014268


In [8]:
metadata = pd.read_csv(METADATA_PATH, dtype=str)
# dtype=strによって文字列として認識
# なぜ文字列とするか？⇒例えば045を数値として読むと45になり、先頭のゼロが消える。
# すると、APIデータの 045 と正しく結合できなくなってしまう

metadata

,class_id,class_name,code,name,level,unit
0,cat01,調査項目 (金融業、保険業以外の業種),001,母集団(当期末),1,社
1,cat01,調査項目 (金融業、保険業以外の業種),002,現金・預金(当期末流動資産),1,百万円
2,cat01,調査項目 (金融業、保険業以外の業種),003,受取手形(当期末流動資産),1,百万円
3,cat01,調査項目 (金融業、保険業以外の業種),004,売掛金(当期末流動資産),1,百万円
4,cat01,調査項目 (金融業、保険業以外の業種),005,株式(当期末流動資産),1,百万円
...,...,...,...,...,...,...
371,time,年 期,20210,2021年度,1,NaN
372,time,年 期,20220,2022年度,1,NaN
373,time,年 期,20230,2023年度,1,NaN
374,time,年 期,20240,2024年度,1,NaN


In [9]:
def create_name_mapping(class_id: str) -> dict[str, str]:
    target = metadata.loc[
        metadata["class_id"] == class_id,# 引数のclass_idと同じclass_id
        ["code", "name"],
    ].drop_duplicates()

    return dict(zip(target["code"], target["name"]))
# zip()は複数のイテラブル（リストやタプルなど）の要素を同じインデックス（順番）ごとにまとめてペアを作る組み込み関数

item_mapping = create_name_mapping("cat01")
industry_mapping = create_name_mapping("cat02")
capital_size_mapping = create_name_mapping("cat03")
time_mapping = create_name_mapping("time")

print(item_mapping)
print(industry_mapping)
print(capital_size_mapping)
print(time_mapping)

{'001': '母集団(当期末)', '002': '現金・預金(当期末流動資産)', '003': '受取手形(当期末流動資産)', '004': '売掛金(当期末流動資産)', '005': '株式(当期末流動資産)', '006': '公社債(当期末流動資産)', '007': 'その他の有価証券(当期末流動資産)', '008': '製品又は商品(当期末流動資産)', '009': '仕掛品(当期末流動資産)', '010': '原材料・貯蔵品(当期末流動資産)', '011': 'その他流動資産(当期末流動資産)', '012': '土地(当期末固定資産)', '013': '建設仮勘定(当期末固定資産)', '014': 'その他の有形固定資産(当期末固定資産)', '015': 'ソフトウェアを除く無形固定資産(当期末固定資産)', '016': 'ソフトウェア(当期末固定資産)', '217': '無形固定資産計(当期末固定資産)', '017': '株式(当期末固定資産)', '018': '公社債(当期末固定資産)', '019': 'その他の有価証券(当期末固定資産)', '020': 'その他投資(当期末固定資産)', '021': '繰延資産(当期末)', '022': '資産合計(当期末)', '023': '受取手形割引残高(当期末)', '024': '支払手形(当期末流動負債)', '025': '買掛金(当期末流動負債)', '026': '金融機関借入金(当期末流動負債)', '027': 'その他の借入金(当期末流動負債)', '028': '引当金(当期末流動負債)', '029': 'その他流動負債(当期末流動負債)', '030': '社債(当期末固定負債)', '031': '金融機関借入金(当期末固定負債)', '032': 'その他の借入金(当期末固定負債)', '033': '引当金(当期末固定負債)', '034': 'その他固定負債(当期末固定負債)', '035': '特別法上の準備金(当期末)', '036': '資本金(当期末)', '037': '資本準備金(当期末)', '039': 'その他資本剰余金(当期末)', '038': '利益準備金(当期末)', '040': '積立金(当期末)', 

In [13]:
df["item_name"] = df["item_code"].map(item_mapping) 
#cat01のcodeと一致するnameを新しい列item_nameとして作る
df["industry_name"] = df["industry_code"].map(industry_mapping)
df["capital_size_name"] = df["capital_size_code"].map(capital_size_mapping)
df["fiscal_year"] = df["time_code"].str[:4].astype(int) # 例えば20250などを2025に変換
df["amount"] = pd.to_numeric(df["amount"], errors="raise") 
# 文字列になっている金額を数値に変換、エラーがあればエラー出て停止

print(df.shape)
df.head()

(540, 10)


,fiscal_year,time_code,industry_code,industry_name,capital_size_code,capital_size_name,item_code,item_name,amount,unit
20,2016,20160,108,製造業,16,1千万円未満,022,資産合計(当期末),10273532,百万円
110,2016,20160,108,製造業,16,1千万円未満,045,売上高(当期末),12488919,百万円
200,2016,20160,108,製造業,16,1千万円未満,051,経常利益(当期末),438726,百万円
290,2016,20160,108,製造業,16,1千万円未満,066,従業員給与(当期末),2169512,百万円
380,2016,20160,108,製造業,16,1千万円未満,085,実物投資(当期末資金需給),544406,百万円


In [14]:
output_columns = [
    "fiscal_year",
    "time_code",
    "industry_code",
    "industry_name",
    "capital_size_code",
    "capital_size_name",
    "item_code",
    "item_name",
    "amount",
    "unit",
]

df = df[output_columns].sort_values(
    ["fiscal_year", "industry_code", "capital_size_code", "item_code"]
)
print(df.shape)
df.head()

(540, 10)


,fiscal_year,time_code,industry_code,industry_name,capital_size_code,capital_size_name,item_code,item_name,amount,unit
20,2016,20160,108,製造業,16,1千万円未満,022,資産合計(当期末),10273532,百万円
110,2016,20160,108,製造業,16,1千万円未満,045,売上高(当期末),12488919,百万円
200,2016,20160,108,製造業,16,1千万円未満,051,経常利益(当期末),438726,百万円
290,2016,20160,108,製造業,16,1千万円未満,066,従業員給与(当期末),2169512,百万円
380,2016,20160,108,製造業,16,1千万円未満,085,実物投資(当期末資金需給),544406,百万円


In [15]:
negative_records = df[df["amount"] < 0]

if not negative_records.empty:
    print("\n負の値を含む財務項目:")
    print(
        negative_records[["item_code", "item_name"]]
        .drop_duplicates()
        .to_string(index=False)
    )


負の値を含む財務項目:
item_code     item_name
      051     経常利益(当期末)
      157      純資産(当期末)
      085 実物投資(当期末資金需給)
